# 01. Load & Clean

매출 CSV를 읽어 날짜·결측치·이상치를 점검하고 `sales_clean.csv`로 저장합니다.

샘플 데이터: `../data/sample_sales.csv` (24개월 × 3 상품)

본인 데이터: `INPUT_PATH`를 `../data/sales.csv`로 바꾸세요.

In [ ]:
!pip install -r ../../../requirements.txt

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data')
INPUT_PATH = DATA_DIR / 'sample_sales.csv'   # ← 본인 데이터로 바꿀 때 'sales.csv'
OUTPUT_PATH = DATA_DIR / 'sales_clean.csv'

df = pd.read_csv(INPUT_PATH, parse_dates=['date'])
print(f'{len(df)} rows, {df["product_id"].nunique()} products, '
      f'{df["date"].min().date()} ~ {df["date"].max().date()}')
df.head()

## 결측치 점검

In [ ]:
missing = df.isna().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'None')

## 이상치 점검 (3-시그마)

각 product_id별로 3-시그마를 벗어나는 행을 표시. 자동 제거하지 않습니다 — 학생이 검토.

In [ ]:
def flag_outliers(g):
    mu = g['units_sold'].mean()
    sd = g['units_sold'].std() or 1.0
    g['outlier'] = (g['units_sold'] - mu).abs() > 3 * sd
    return g

df = df.groupby('product_id', group_keys=False).apply(flag_outliers)
print(f'Outliers flagged: {df["outlier"].sum()} rows')
df[df['outlier']].head()

## 날짜 누락 점검

월별 데이터를 가정 — 상품마다 모든 월이 있는지.

In [ ]:
all_months = pd.date_range(df['date'].min(), df['date'].max(), freq='MS')
for pid, g in df.groupby('product_id'):
    missing_dates = set(all_months) - set(g['date'])
    if missing_dates:
        print(f'{pid}: missing {len(missing_dates)} months')
    else:
        print(f'{pid}: complete ({len(g)} rows)')

## 저장

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved → {OUTPUT_PATH.resolve()}')

## 다음 단계

- `02_analyze_trends.ipynb` 시각화
- 결측·이상치가 많으면 Claude Desktop의 `data_diagnosis(si)` 프롬프트로 진단 권장